# Tutorial: Scorey Eval DB Walkthrough

Audience:
- This is for the imagineer side of the project: someone who wants to inspect the first eval lane without having to reverse-engineer the runtime.

Prerequisites:
- Run `make install` once.
- Open this notebook from the repo root.

Learning goals:
- By the end, you can initialize the eval database.
- By the end, you can record local Scorey rounds into it.
- By the end, you can run the Beta 1.0 picks gate against stored rows.
- By the end, you can inspect counts and apply a binary judgment.


## Outline

1. Point at a safe scratch database
2. Initialize the schema
3. Record a few local rounds
4. Inspect rows and counts
5. Run the Beta 1.0 picks gate
6. Judge one row
7. Map the notebook back to the CLI surface


In [ ]:
from __future__ import annotations

from scorey.eval_db import counts, default_eval_db_path, init_db, judge_output, list_outputs, record_round_state
from scorey.eval_gates import beta_1_pass_pairs, evaluate_beta_1
from scorey.pipeline import build_local_round_state, compose_round

MAIN_DB_PATH = default_eval_db_path()
SCRATCH_DB_PATH = MAIN_DB_PATH.with_name("scorey-eval-walkthrough.sqlite")
SCRATCH_DB_PATH


## Step 1 - Start with a clean scratch database

This notebook uses a scratch SQLite file so you can rerun cells without touching the main operator database at `.local/evals.sqlite`.


In [ ]:
if SCRATCH_DB_PATH.exists():
    SCRATCH_DB_PATH.unlink()

init_db(SCRATCH_DB_PATH)
SCRATCH_DB_PATH.exists(), SCRATCH_DB_PATH


## Step 2 - Record a few local rounds

The helper below uses the same runtime pieces as the CLI local mode:

- `build_local_round_state(...)`
- `compose_round(...)`
- `record_round_state(...)`


In [ ]:
def record_local_round(user_pick: str, *, scorey_score: int = 1) -> int:
    round_state = build_local_round_state(user_pick, scorey_score=scorey_score)
    round_text = compose_round(round_state)
    return record_round_state(
        SCRATCH_DB_PATH,
        round_state,
        round_text,
        source_mode="local",
        model="local-fixture",
    )

recorded_ids = [
    record_local_round("rock", scorey_score=1),
    record_local_round("paper", scorey_score=2),
    record_local_round("scissors", scorey_score=3),
]
recorded_ids


## Step 3 - Inspect the stored rows

Each output row keeps the round text plus the small routing metadata that makes the row legible without rereading the whole runtime.


In [ ]:
rows = list_outputs(SCRATCH_DB_PATH, limit=10)
[dict(row) for row in rows]


## Step 4 - Check the verdict counts

Before judgment, every row should still be pending.


In [ ]:
counts(SCRATCH_DB_PATH)


## Step 5 - Run the Beta 1.0 picks gate

Beta 1.0 only judges the pick pair in `scorey_pick, user_pick` order.

Pass pairs:
- reverse gameplay routes
- same-pick loophole routes

Fail:
- every other pair


In [ ]:
row_results = []
for row in rows:
    result = evaluate_beta_1(row["user_pick"], row["scorey_pick"])
    row_results.append(
        {
            "scorey_pick": result.scorey_pick,
            "user_pick": result.user_pick,
            "verdict": result.verdict,
            "reason": result.reason,
        }
    )

{
    "pass_pairs": beta_1_pass_pairs(),
    "row_results": row_results,
}


## Step 6 - Apply one binary judgment

Judgments are append-only history in `eval_judgments`, and the latest verdict is mirrored onto the output row for quick listing.


In [ ]:
judge_output(
    SCRATCH_DB_PATH,
    recorded_ids[0],
    "pass",
    "pick-specific and easy to read",
)

counts(SCRATCH_DB_PATH)


In [ ]:
[dict(row) for row in list_outputs(SCRATCH_DB_PATH, limit=10)]


## Step 7 - Map this back to the CLI

Notebook lane:

- uses a scratch file for safety
- calls the tracked Python module functions directly

Operator lane:

- `make eval-init`
- `make eval-list EVAL_LIMIT=10`
- `make eval-beta1 EVAL_LIMIT=10`

When you are ready to use the real operator database, swap `SCRATCH_DB_PATH` for `MAIN_DB_PATH` in the setup cell above.
